In [0]:
from pyspark.sql.functions import col, when

In [0]:
source_path = "oftalmo_sus.01_bronze.bronze_dim_sigtap"
target_path = "oftalmo_sus.02_silver.tb_dim_sigtap_oftalmo"

In [0]:
df_bronze_sigtap = spark.table(source_path)

In [0]:
# Colunas que não podem ser nulas no dicionário
colunas_criticas_dicionario = ["CO_PROCEDIMENTO", "NO_PROCEDIMENTO"]

df_silver_sigtap = (
    df_bronze_sigtap
    # Filtro de Oftalmologia pelos prefixos do SUS
    .filter(
        col("CO_PROCEDIMENTO").startswith("0405") |    # Cirurgias do aparelho da visão
        col("CO_PROCEDIMENTO").startswith("030305") |  # Tratamentos clínicos em oftalmologia
        col("CO_PROCEDIMENTO").startswith("021106")    # Exames oftalmológicos
    )
    # Controle de Qualidade (Drop Nulls)
    .dropna(subset=colunas_criticas_dicionario)
    
    # Traduzindo a Complexidade
    .withColumn(
        "DESC_COMPLEXIDADE", 
        when(col("TP_COMPLEXIDADE") == "1", "Atenção Básica")
        .when(col("TP_COMPLEXIDADE") == "2", "Média Complexidade")
        .when(col("TP_COMPLEXIDADE") == "3", "Alta Complexidade")
        .otherwise("Não Classificado")
    )
    
    # Selecionando apenas o que importa para o modelo dimensional
    .select(
        "CO_PROCEDIMENTO", 
        "NO_PROCEDIMENTO", 
        "DESC_COMPLEXIDADE"
    )
)

display(df_silver_sigtap.limit(15))

In [0]:
print(f"Total de procedimentos oftamológicos válidos: {df_silver_sigtap.count()}")

In [0]:
(
    df_silver_sigtap.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_path)
)